In [1]:
import pandas as pd

# Load dataset splits
train = pd.read_csv("train.txt", sep="\t", names=["head","relation","tail"])
valid = pd.read_csv("valid.txt", sep="\t", names=["head","relation","tail"])
test = pd.read_csv("test.txt", sep="\t", names=["head","relation","tail"])

print("Train shape:", train.shape)
print("Valid shape:", valid.shape)
print("Test shape:", test.shape)

Train shape: (120656, 3)
Valid shape: (17535, 3)
Test shape: (20466, 3)


In [2]:
# Remove nulls
train = train.dropna().drop_duplicates().reset_index(drop=True)
valid = valid.dropna().drop_duplicates().reset_index(drop=True)
test = test.dropna().drop_duplicates().reset_index(drop=True)

In [3]:
def feature_engineering(df):

    # Length features
    df["head_length"] = df["head"].apply(len)
    df["tail_length"] = df["tail"].apply(len)
    df["relation_length"] = df["relation"].apply(len)

    # Word count
    df["head_word_count"] = df["head"].apply(lambda x: len(x.split("_")))
    df["tail_word_count"] = df["tail"].apply(lambda x: len(x.split("_")))
    df["relation_word_count"] = df["relation"].apply(lambda x: len(x.split("_")))

    # Uppercase count
    df["head_upper"] = df["head"].apply(lambda x: sum(c.isupper() for c in x))
    df["tail_upper"] = df["tail"].apply(lambda x: sum(c.isupper() for c in x))
    df["relation_upper"] = df["relation"].apply(lambda x: sum(c.isupper() for c in x))

    # Digit count
    df["head_digit_count"] = df["head"].apply(lambda x: sum(c.isdigit() for c in x))
    df["tail_digit_count"] = df["tail"].apply(lambda x: sum(c.isdigit() for c in x))
    df["relation_digit_count"] = df["relation"].apply(lambda x: sum(c.isdigit() for c in x))

    # Has number
    df["head_has_number"] = df["head_digit_count"].apply(lambda x: 1 if x > 0 else 0)
    df["tail_has_number"] = df["tail_digit_count"].apply(lambda x: 1 if x > 0 else 0)
    df["relation_has_number"] = df["relation_digit_count"].apply(lambda x: 1 if x > 0 else 0)

    return df

train = feature_engineering(train)
valid = feature_engineering(valid)
test = feature_engineering(test)

In [4]:
# Compute degrees using training data only
head_degree = train["head"].value_counts()
tail_degree = train["tail"].value_counts()
relation_freq = train["relation"].value_counts()

def add_degree_features(df):
    df["head_degree"] = df["head"].map(head_degree).fillna(0)
    df["tail_degree"] = df["tail"].map(tail_degree).fillna(0)
    df["relation_frequency"] = df["relation"].map(relation_freq).fillna(0)
    df["degree_difference"] = df["head_degree"] - df["tail_degree"]
    df["head_tail_length_diff"] = df["head_length"] - df["tail_length"]
    df["head_tail_length_sum"] = df["head_length"] + df["tail_length"]
    return df

train = add_degree_features(train)
valid = add_degree_features(valid)
test = add_degree_features(test)

In [5]:
from sklearn.preprocessing import LabelEncoder

# Combine all data for fitting encoders
all_heads = pd.concat([train["head"], valid["head"], test["head"]])
all_tails = pd.concat([train["tail"], valid["tail"], test["tail"]])
all_relations = pd.concat([train["relation"], valid["relation"], test["relation"]])

# Create encoders
le_head = LabelEncoder()
le_tail = LabelEncoder()
le_relation = LabelEncoder()

# Fit on all data
le_head.fit(all_heads)
le_tail.fit(all_tails)
le_relation.fit(all_relations)

# Transform each dataset
for df in [train, valid, test]:
    df["head_enc"] = le_head.transform(df["head"])
    df["tail_enc"] = le_tail.transform(df["tail"])
    df["relation_enc"] = le_relation.transform(df["relation"])

In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Select numeric columns except target
feature_cols = train.select_dtypes(include=["int64","float64"]).columns
feature_cols = feature_cols.drop("relation_enc")

# Fit on train only
train[feature_cols] = scaler.fit_transform(train[feature_cols])

# Transform valid & test
valid[feature_cols] = scaler.transform(valid[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

In [7]:
X_train = train.drop(columns=["head","tail","relation","relation_enc"])
y_train = train["relation_enc"]

X_valid = valid.drop(columns=["head","tail","relation","relation_enc"])
y_valid = valid["relation_enc"]

X_test = test.drop(columns=["head","tail","relation","relation_enc"])
y_test = test["relation_enc"]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (120655, 23)
Testing shape: (20466, 23)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, pred_lr))

Logistic Regression Accuracy: 0.7560343985146096


In [9]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, pred_rf))

Random Forest Accuracy: 0.9922798788234144


In [10]:
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

print(classification_report(y_test, pred_rf, zero_division=0))

              precision    recall  f1-score   support

           1       0.95      0.90      0.92        20
           2       0.88      0.93      0.90        15
           3       1.00      1.00      1.00       858
           4       1.00      1.00      1.00       217
           5       1.00      1.00      1.00       323
           6       1.00      1.00      1.00       317
           7       1.00      1.00      1.00       121
           8       0.94      1.00      0.97        16
           9       1.00      1.00      1.00      1067
          10       1.00      1.00      1.00       214
          11       1.00      1.00      1.00       132
          12       1.00      1.00      1.00        41
          13       1.00      1.00      1.00       118
          14       0.92      1.00      0.96        24
          15       1.00      1.00      1.00        10
          16       1.00      0.93      0.96        27
          17       1.00      1.00      1.00        48
          18       1.00    

In [11]:
!pip install neo4j tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 6.6 MB/s eta 0:00:00


In [12]:
from neo4j import GraphDatabase

uri = "neo4j+s://71995882.databases.neo4j.io"
username = "71995882"
password = "UuSgt9NWfJY0_4NCzsQlsbSjnr662VgEkxEly5gV2vY"

driver = GraphDatabase.driver(uri, auth=(username, password))

print("Connected successfully!")

Connected successfully!


In [13]:
df.head()
print(len(df))

20466


In [14]:
from tqdm import tqdm

def insert_batch(tx, batch):
    query = """
    UNWIND $rows AS row
    MERGE (h:Entity {name: row.head})
    MERGE (t:Entity {name: row.tail})
    MERGE (h)-[:RELATION {type: row.relation}]->(t)
    """
    tx.run(query, rows=batch)

batch_size = 1000
records = df.to_dict("records")

with driver.session() as session:
    for i in tqdm(range(0, len(records), batch_size)):
        batch = records[i:i+batch_size]
        session.execute_write(insert_batch, batch)

print("Data inserted successfully!")

100%|██████████| 21/21 [03:16<00:00,  9.37s/it]

Data inserted successfully!


In [15]:
!pip install spacy neo4j
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 808.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 23.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
import spacy

# Load transformer-based NER model
nlp = spacy.load("en_core_web_trf")

print("Model Loaded Successfully")

Model Loaded Successfully


In [17]:
text = """
Barack Obama was born in Hawaii.
He was the President of the United States.
Microsoft was founded by Bill Gates in Albuquerque.
Google is headquartered in California.
"""

In [18]:
doc = nlp(text)

entities = []

for ent in doc.ents:
    print(ent.text, "->", ent.label_)
    entities.append((ent.text, ent.label_))

Barack Obama -> PERSON
Hawaii -> GPE
the United States -> GPE
Microsoft -> ORG
Bill Gates -> PERSON
Albuquerque -> GPE
Google -> ORG
California -> GPE


In [19]:
entities = list(set(entities))
print(entities)

[('Barack Obama', 'PERSON'), ('Bill Gates', 'PERSON'), ('the United States', 'GPE'), ('Albuquerque', 'GPE'), ('California', 'GPE'), ('Google', 'ORG'), ('Hawaii', 'GPE'), ('Microsoft', 'ORG')]


In [20]:
from neo4j import GraphDatabase

uri = "neo4j+s://71995882.databases.neo4j.io"
username = "71995882"
password = "UuSgt9NWfJY0_4NCzsQlsbSjnr662VgEkxEly5gV2vY"

driver = GraphDatabase.driver(uri, auth=(username, password))

In [21]:
def create_entity(tx, name, label):
    query = f"MERGE (e:{label} {{name: $name}})"
    tx.run(query, name=name)

with driver.session() as session:
    for name, label in entities:
        session.execute_write(create_entity, name, label)

print("Entities Stored Successfully")

Entities Stored Successfully


In [22]:
def create_relationship(tx, e1, e2):
    query = """
    MATCH (a {name: $e1})
    MATCH (b {name: $e2})
    MERGE (a)-[:RELATED_TO]->(b)
    """
    tx.run(query, e1=e1, e2=e2)

sentences = list(doc.sents)

with driver.session() as session:
    for sent in sentences:
        sent_entities = [ent.text for ent in sent.ents]
        for i in range(len(sent_entities)):
            for j in range(i+1, len(sent_entities)):
                session.execute_write(
                    create_relationship,
                    sent_entities[i],
                    sent_entities[j]
                )

print("Relationships Created Successfully")

Relationships Created Successfully


In [23]:
!pip install sentence-transformers pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 20.6 MB/s eta 0:00:00


In [24]:
from sentence_transformers import SentenceTransformer
import pinecone
import numpy as np

In [27]:
from pinecone import Pinecone, ServerlessSpec

In [28]:
pc = Pinecone(api_key="pcsk_2HwvL3_PuG7ed7Zp3c6hTykLV5ojHxz8wa7ikPusqbSbm8Vz6bjXyGnsqEcgZKuofyVZ4N")

In [37]:
from pinecone import Pinecone, ServerlessSpec

# Initialize your client
pc = Pinecone(api_key="pcsk_2HwvL3_PuG7ed7Zp3c6hTykLV5ojHxz8wa7ikPusqbSbm8Vz6bjXyGnsqEcgZKuofyVZ4N")

index_name = "semantic-search-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        index=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

In [38]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
documents = [
    "Barack Obama was the President of the United States",
    "Microsoft was founded by Bill Gates",
    "Google is headquartered in California",
    "Tesla was founded by Elon Musk",
    "Amazon is an e-commerce company"
]

In [40]:
embeddings = model.encode(documents)

In [42]:
from pinecone import Pinecone

# 1. Initialize Pinecone with your API Key
pc = Pinecone(api_key="pcsk_2HwvL3_PuG7ed7Zp3c6hTykLV5ojHxz8wa7ikPusqbSbm8Vz6bjXyGnsqEcgZKuofyVZ4N")

# 2. Define the 'index' variable by connecting to your specific index name
# This is the step that is likely missing!
index = pc.Index("semantic-search-index")

# Now your existing code will work:
# index.upsert(vectors)

In [43]:
vectors = []

for i, emb in enumerate(embeddings):
    vectors.append(
        (str(i), emb.tolist())
    )

index.upsert(vectors)

print("Embeddings Stored in Pinecone")

Embeddings Stored in Pinecone


In [44]:
query = "Who founded Microsoft?"

query_embedding = model.encode([query])

results = index.query(
    vector=query_embedding[0].tolist(),
    top_k=3
)

print(results)

QueryResponse(matches=[{'id': '1', 'score': 0.903079033, 'values': []}, {'id': '3', 'score': 0.370456725, 'values': []}, {'id': '2', 'score': 0.276344299, 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sun, 29 Mar 2026 14:49:32 GMT', 'content-type': 'application/json', 'content-length': '194', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '95', 'x-envoy-upstream-service-time': '95', 'x-pinecone-response-duration-ms': '98', 'grpc-status': '0', 'server': 'envoy'}})


In [45]:
for match in results['matches']:
    idx = int(match['id'])
    print(documents[idx])

Microsoft was founded by Bill Gates
Tesla was founded by Elon Musk
Google is headquartered in California
